# 覓聲 MeetSum — 重點摘要 LoRA 微調
訓練答案 = Claude 依覓聲摘要規則寫的會議摘要。原版 qwen3.8:27b 不動。
從上到下依序執行；最後一格「關機」請確認結果都看到了再執行。

In [ ]:
# 0. 環境與檔案（程式、訓練資料都內嵌在 notebook 裡）
import os, sys, json, gzip, base64, subprocess, shutil
W = '/content/sum_ft'; os.makedirs(f'{W}/data', exist_ok=True); os.makedirs(f'{W}/out', exist_ok=True)
for n, s in {'lm_utils.py': '"""訓練與評測共用：組出和推論時完全相同的 prompt 文字（關閉 thinking）。"""\nfrom __future__ import annotations\n\n\ndef chat_prompt(tok, system: str, user: str) -> str:\n    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]\n    try:\n        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)\n    except TypeError:\n        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)\n\n\ndef end_text(tok) -> str:\n    """回答結尾：Qwen 系列是 <|im_end|>。"""\n    eos = tok.eos_token or "<|im_end|>"\n    return "<|im_end|>" if "<|im_end|>" in tok.get_vocab() else eos\n\n\ndef load_tokenizer(path: str):\n    from transformers import AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(path, trust_remote_code=True)\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    return tok\n\n\ndef load_model(path: str, adapter: str | None = None, dtype: str = "bfloat16", load_4bit: bool = False):\n    import torch\n    from transformers import AutoConfig\n    cfg = AutoConfig.from_pretrained(path, trust_remote_code=True)\n    kw = {"dtype": getattr(torch, dtype), "device_map": "auto", "trust_remote_code": True, "low_cpu_mem_usage": True}\n    if load_4bit:\n        from transformers import BitsAndBytesConfig\n        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n                                                       bnb_4bit_compute_dtype=getattr(torch, dtype),\n                                                       bnb_4bit_use_double_quant=True)\n    if getattr(cfg, "vision_config", None) is not None:\n        from transformers import AutoModelForImageTextToText as Auto\n    else:\n        from transformers import AutoModelForCausalLM as Auto\n    model = Auto.from_pretrained(path, **kw)\n    if adapter:\n        from peft import PeftModel\n        model = PeftModel.from_pretrained(model, adapter)\n    model.eval()\n    return model\n', 'train_lora.py': '"""步驟 3（Colab GPU）：LoRA 微調 Qwen3.8-27B 的錯字校正能力。\n\n- 只訓練語言模型的注意力與 MLP 投影層（LoRA），原始權重不動；視覺部分完全不碰。\n- loss 只算「模型的回答」那段，prompt（系統指令＋逐字稿）不算。\n- checkpoint 存到 --out（請放在 Google Drive），斷線後重跑會自動接續。\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport random\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport lm_utils as L  # noqa: E402\n\n# 全注意力層：q/k/v/o；線性注意力層（Gated DeltaNet）：in_proj_qkv / in_proj_z / out_proj；每層的 MLP。\n_MODS = (r"(self_attn\\.(q_proj|k_proj|v_proj|o_proj)|linear_attn\\.(in_proj_qkv|in_proj_z|out_proj)"\n         r"|mlp\\.(gate_proj|up_proj|down_proj))$")\nTARGET = r"^(base_model\\.model\\.)?model\\.(language_model\\.)?layers\\.\\d+\\." + _MODS\n\n\ndef encode(tok, ex: dict, max_len: int):\n    prompt = L.chat_prompt(tok, ex["system"], ex["user"])\n    answer = ex["target"] + L.end_text(tok) + "\\n"\n    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]\n    a_ids = tok(answer, add_special_tokens=False)["input_ids"]\n    if len(p_ids) + len(a_ids) > max_len:\n        return None\n    return {"input_ids": p_ids + a_ids, "labels": [-100] * len(p_ids) + a_ids}\n\n\nclass Collator:\n    def __init__(self, pad_id: int):\n        self.pad_id = pad_id\n\n    def __call__(self, feats):\n        import torch\n        n = max(len(f["input_ids"]) for f in feats)\n        ids = torch.full((len(feats), n), self.pad_id, dtype=torch.long)\n        lab = torch.full((len(feats), n), -100, dtype=torch.long)\n        att = torch.zeros((len(feats), n), dtype=torch.long)\n        for i, f in enumerate(feats):\n            k = len(f["input_ids"])\n            ids[i, :k] = torch.tensor(f["input_ids"])\n            lab[i, :k] = torch.tensor(f["labels"])\n            att[i, :k] = 1\n        return {"input_ids": ids, "labels": lab, "attention_mask": att}\n\n\ndef load(path: Path, kinds: set[str] | None = None) -> list[dict]:\n    rows = [json.loads(x) for x in path.open(encoding="utf-8") if x.strip()]\n    return [r for r in rows if kinds is None or r["kind"] in kinds]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--base", required=True)\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--epochs", type=float, default=2.0)\n    ap.add_argument("--lr", type=float, default=1e-4)\n    ap.add_argument("--rank", type=int, default=16)\n    ap.add_argument("--alpha", type=int, default=32)\n    ap.add_argument("--max-len", type=int, default=6144)\n    ap.add_argument("--grad-accum", type=int, default=16)\n    ap.add_argument("--save-steps", type=int, default=20)\n    ap.add_argument("--dev-limit", type=int, default=48)\n    ap.add_argument("--load-4bit", action="store_true", help="顯示卡記憶體不夠時改用 QLoRA")\n    ap.add_argument("--max-steps", type=int, default=-1, help="測試用")\n    ap.add_argument("--max-train", type=int, default=1200, help="訓練樣本上限（控制 Colab 時數）；0 = 全部")\n    a = ap.parse_args()\n\n    # Colab 內建的舊版 torchao（0.10）會讓新版 peft 直接報錯；本專案用不到它，有舊版就移除\n    try:\n        import importlib.metadata as md\n        from packaging.version import Version\n        if Version(md.version("torchao")) < Version("0.16.0"):\n            import subprocess\n            print("移除不相容的舊版 torchao", md.version("torchao"), flush=True)\n            subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)\n    except Exception:\n        pass\n\n    import torch\n    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n    from transformers import Trainer, TrainingArguments, trainer_utils\n\n    out = Path(a.out)\n    out.mkdir(parents=True, exist_ok=True)\n    tok = L.load_tokenizer(a.base)\n\n    train_rows = load(Path(a.data) / "sft_train.jsonl")\n    dev_rows = load(Path(a.data) / "sft_dev.jsonl")\n    random.Random(0).shuffle(dev_rows)\n    random.Random(1).shuffle(train_rows)\n    if a.max_train:\n        train_rows = train_rows[:a.max_train]\n    train = [e for e in (encode(tok, r, a.max_len) for r in train_rows) if e]\n    dev = [e for e in (encode(tok, r, a.max_len) for r in dev_rows) if e][:a.dev_limit]\n    lens = sorted(len(e["input_ids"]) for e in train)\n    print(f"訓練 {len(train)} 筆（超過 {a.max_len} token 丟掉 {len(train_rows) - len(train)} 筆）；"\n          f"dev {len(dev)} 筆；長度中位數 {lens[len(lens) // 2]}、最大 {lens[-1]}", flush=True)\n\n    model = L.load_model(a.base, load_4bit=a.load_4bit)\n    model.config.use_cache = False\n    if a.load_4bit:\n        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n    else:\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        model.enable_input_require_grads()\n    lcfg = LoraConfig(r=a.rank, lora_alpha=a.alpha, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",\n                      target_modules=TARGET)\n    model = get_peft_model(model, lcfg)\n    model.print_trainable_parameters()\n    names = [n for n, _ in model.named_modules() if n.endswith("lora_A")]\n    print("LoRA 掛在", len(names), "個模組，例如", names[:3], flush=True)\n    if not names:\n        raise RuntimeError("LoRA 沒有掛上任何模組，請檢查 TARGET 正規表示式")\n\n    steps_per_epoch = math.ceil(len(train) / a.grad_accum)\n    total_steps = a.max_steps if a.max_steps > 0 else math.ceil(steps_per_epoch * a.epochs)\n    args = TrainingArguments(\n        output_dir=str(out / "checkpoints"),\n        per_device_train_batch_size=1, per_device_eval_batch_size=1,\n        gradient_accumulation_steps=a.grad_accum,\n        num_train_epochs=a.epochs, max_steps=a.max_steps,\n        learning_rate=a.lr, lr_scheduler_type="cosine", warmup_steps=max(1, int(total_steps * 0.03)), weight_decay=0.0,\n        bf16=torch.cuda.is_available(), logging_steps=2,\n        eval_strategy="steps" if dev else "no", eval_steps=a.save_steps, save_strategy="steps", save_steps=a.save_steps,\n        save_total_limit=3, load_best_model_at_end=bool(dev), metric_for_best_model="eval_loss", greater_is_better=False,\n        report_to="none", remove_unused_columns=False, dataloader_num_workers=0, label_names=["labels"],\n        gradient_checkpointing=False,  # 上面已經自己開好\n        seed=42,\n    )\n    print(f"每個 epoch {steps_per_epoch} 步，共 {a.epochs} 個 epoch", flush=True)\n    trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=dev or None, data_collator=Collator(tok.pad_token_id))\n    last = trainer_utils.get_last_checkpoint(str(out / "checkpoints")) if (out / "checkpoints").exists() else None\n    if last:\n        print("接續上次的 checkpoint：", last, flush=True)\n    trainer.train(resume_from_checkpoint=last)\n    final = out / "final"\n    trainer.model.save_pretrained(str(final))\n    tok.save_pretrained(str(final))\n    hist = [h for h in trainer.state.log_history]\n    (out / "train_log.json").write_text(json.dumps(hist, ensure_ascii=False, indent=1), encoding="utf-8")\n    (out / "train_args.json").write_text(json.dumps(vars(a), ensure_ascii=False, indent=1), encoding="utf-8")\n    print("完成，adapter 存在", final, "（最佳 checkpoint：", trainer.state.best_model_checkpoint, "）")\n\n\nif __name__ == "__main__":\n    main()\n', 'eval_sum.py': '"""摘要評測：原版（不掛 LoRA）vs 微調版（掛 LoRA），和我寫的參考摘要比較。\n\n指標（每場會議算完取平均）：\n- json_ok：輸出是合法 JSON 且結構正確\n- quote_ok：引用原文確實出現在逐字稿的比例（有沒有捏造）\n- count_diff：四類項目（報告／討論／決議／待辦）數量和參考摘要的平均差距（越小越好）\n- topic_f1：報告＋討論的議題，和參考摘要逐一配對的 F1（字元相似度 ≥ 0.5 算配對成功）\n- action_f1：待辦事項和參考摘要配對的 F1\n- text_sim：整份摘要文字（headline＋各項摘要＋待辦）和參考摘要的字元 ROUGE-L F1\n- headline_sim：一句話總結的字元 ROUGE-L F1\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport difflib\nimport json\nimport re\nimport sys\nimport time\nfrom pathlib import Path\n\nCATS = ["reports", "discussions", "decisions", "action_items"]\n\n\ndef parse(text: str):\n    t = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S).strip()\n    t = re.sub(r"^```(?:json)?\\s*|\\s*```$", "", t)\n    try:\n        j = json.loads(t)\n    except Exception:\n        m = re.search(r"\\{.*\\}", t, flags=re.S)\n        if not m:\n            return None\n        try:\n            j = json.loads(m.group(0))\n        except Exception:\n            return None\n    if not isinstance(j, dict) or not isinstance(j.get("meeting_content"), dict):\n        return None\n    return j\n\n\ndef _items(j, cat):\n    v = (j or {}).get("meeting_content", {}).get(cat) or []\n    return [x for x in v if isinstance(x, dict)]\n\n\ndef lcs_f1(a: str, b: str) -> float:\n    if not a or not b:\n        return 0.0\n    a, b = a[:1500], b[:1500]\n    prev = [0] * (len(b) + 1)\n    for x in a:\n        cur = [0]\n        for j, y in enumerate(b, 1):\n            cur.append(prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1]))\n        prev = cur\n    l = prev[-1]\n    p, r = l / len(a), l / len(b)\n    return 2 * p * r / (p + r) if p + r else 0.0\n\n\ndef match_f1(pred: list[str], ref: list[str], thr=0.5) -> float:\n    if not pred and not ref:\n        return 1.0\n    if not pred or not ref:\n        return 0.0\n    used, tp = set(), 0\n    for p in pred:\n        best, bi = 0, None\n        for i, r in enumerate(ref):\n            if i in used:\n                continue\n            s = difflib.SequenceMatcher(None, p.lower(), r.lower()).ratio()\n            if s > best:\n                best, bi = s, i\n        if best >= thr:\n            used.add(bi)\n            tp += 1\n    pr, rc = tp / len(pred), tp / len(ref)\n    return 2 * pr * rc / (pr + rc) if pr + rc else 0.0\n\n\ndef flat_text(j) -> str:\n    parts = [str((j or {}).get("headline") or "")]\n    for cat in CATS:\n        for it in _items(j, cat):\n            parts.append(str(it.get("topic") or ""))\n            parts.append(str(it.get("summary") or it.get("task") or ""))\n            parts += [str(k) for k in (it.get("key_points") or [])]\n    return "".join(parts)\n\n\ndef score(pred_text: str, ref: dict, transcript: dict) -> dict:\n    j = parse(pred_text)\n    if j is None:\n        return {"json_ok": 0, "quote_ok": 0, "count_diff": None, "topic_f1": 0, "action_f1": 0, "text_sim": 0, "headline_sim": 0}\n    seg = {s["segment_id"]: s["text"] for s in transcript["segments"]}\n    q_tot = q_ok = 0\n    for cat in CATS:\n        for it in _items(j, cat):\n            for e in it.get("evidence") or []:\n                if not isinstance(e, dict):\n                    continue\n                q_tot += 1\n                q_ok += bool(e.get("quote")) and str(e.get("quote")) in seg.get(e.get("segment_id"), "")\n    topics = lambda x: [str(i.get("topic") or "") for c in ("reports", "discussions") for i in _items(x, c)]\n    tasks = lambda x: [str(i.get("task") or "") for i in _items(x, "action_items")]\n    return {\n        "json_ok": 1,\n        "quote_ok": q_ok / q_tot if q_tot else 0.0,\n        "count_diff": sum(abs(len(_items(j, c)) - len(_items(ref, c))) for c in CATS) / 4,\n        "topic_f1": match_f1(topics(j), topics(ref)),\n        "action_f1": match_f1(tasks(j), tasks(ref)),\n        "text_sim": lcs_f1(flat_text(j), flat_text(ref)),\n        "headline_sim": lcs_f1(str(j.get("headline") or ""), str(ref.get("headline") or "")),\n    }\n\n\ndef summarize(rows: list[dict]) -> dict:\n    out = {"n": len(rows)}\n    for k in ["json_ok", "quote_ok", "count_diff", "topic_f1", "action_f1", "text_sim", "headline_sim"]:\n        v = [r[k] for r in rows if r[k] is not None]\n        out[k] = round(sum(v) / len(v), 4) if v else None\n    return out\n\n\nclass Shared:\n    def __init__(self, base, adapter, load_4bit=False):\n        sys.path.insert(0, str(Path(__file__).resolve().parent))\n        import lm_utils as L\n        self.L = L\n        self.tok = L.load_tokenizer(base)\n        self.tok.padding_side = "left"\n        self.model = L.load_model(base, adapter, load_4bit=load_4bit)\n        self.use_adapter = True\n\n    def generate(self, batch, max_new_tokens=5120):\n        import torch\n        texts = [self.L.chat_prompt(self.tok, ex["system"], ex["user"]) for ex in batch]\n        enc = self.tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(self.model.device)\n        ctx = contextlib.nullcontext() if self.use_adapter else self.model.disable_adapter()\n        with torch.no_grad(), ctx:\n            gen = self.model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=self.tok.pad_token_id)\n        return self.tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)\n\n\ndef run(data: Path, base: str, adapter: str, out: Path, batch_size: int, load_4bit: bool):\n    out.mkdir(parents=True, exist_ok=True)\n    rows = [json.loads(x) for x in (data / "sft_test.jsonl").open(encoding="utf-8")]\n    m = Shared(base, adapter, load_4bit)\n    report = {}\n    for tag, use in [("base", False), ("ft", True)]:\n        m.use_adapter = use\n        pred_path = out / f"sum_{tag}.predictions.jsonl"\n        done = {}\n        if pred_path.exists():\n            done = {json.loads(l)["id"]: json.loads(l)["reply"] for l in pred_path.open(encoding="utf-8")}\n        todo = [r for r in rows if r["id"] not in done]\n        t0 = time.time()\n        with pred_path.open("a", encoding="utf-8") as f:\n            for i in range(0, len(todo), batch_size):\n                b = todo[i:i + batch_size]\n                for r, rep in zip(b, m.generate(b)):\n                    done[r["id"]] = rep\n                    f.write(json.dumps({"id": r["id"], "reply": rep}, ensure_ascii=False) + "\\n")\n                print(f"  {tag} {i + len(b)}/{len(todo)}  {time.time() - t0:.0f}s", flush=True)\n        per = []\n        for r in rows:\n            s = score(done[r["id"]], json.loads(r["target"]), r["transcript"])\n            s["id"], s["lang"] = r["id"], r["lang"]\n            per.append(s)\n        report[tag] = {"all": summarize(per), "zh": summarize([p for p in per if p["lang"] == "zh"]),\n                       "en": summarize([p for p in per if p["lang"] == "en"]), "per_meeting": per}\n        (out / "sum_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding="utf-8")\n        print(tag, json.dumps(report[tag]["all"], ensure_ascii=False), flush=True)\n    return report\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--base", required=True)\n    ap.add_argument("--adapter", required=True)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--batch-size", type=int, default=4)\n    ap.add_argument("--load-4bit", action="store_true")\n    a = ap.parse_args()\n    run(Path(a.data), a.base, a.adapter, Path(a.out), a.batch_size, a.load_4bit)\n'}.items():
    open(f'{W}/{n}', 'w', encoding='utf-8').write(s)
for n, b in {'sft_train.jsonl': '...', 'sft_dev.jsonl': '...', 'sft_test.jsonl': '...'}.items():
    open(f'{W}/data/{n}', 'wb').write(gzip.decompress(base64.b64decode(b)))
for n in ['sft_train.jsonl', 'sft_dev.jsonl', 'sft_test.jsonl']:
    print(n, sum(1 for _ in open(f'{W}/data/{n}', encoding='utf-8')), '筆')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
GPU_GB = int(subprocess.run(['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout.split()[0]) // 1024
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'peft', 'accelerate', 'bitsandbytes', 'flash-linear-attention'], capture_output=True, text=True)
print('套件安裝', '完成' if r.returncode == 0 else r.stderr[-2000:])
def run(cmd):
    print('$', ' '.join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:
            print(line, end='', flush=True)
        p.wait()
    except KeyboardInterrupt:
        p.kill(); p.wait(); raise
    if p.returncode != 0:
        raise RuntimeError(f'失敗 exit {p.returncode}')

In [ ]:
# 1. 下載原始權重（約 54GB，20–30 分鐘）
BASE = '/content/hf/Qwen3.8-27B'
if not os.path.exists(f'{BASE}/model.safetensors.index.json'):
    from huggingface_hub import snapshot_download
    snapshot_download('Qwen/Qwen3.8-27B', local_dir=BASE, max_workers=16)
print('完成')

In [ ]:
# 2. 訓練 LoRA（摘要的輸入很長，用 4-bit 載入原始權重以節省顯示卡記憶體）
if not os.path.exists(f'{W}/out/run/final/adapter_model.safetensors'):
    run([sys.executable, f'{W}/train_lora.py', '--base', BASE, '--data', f'{W}/data', '--out', f'{W}/out/run',
         '--max-len', '16384', '--load-4bit', '--epochs', '3', '--grad-accum', '4', '--save-steps', '10',
         '--dev-limit', '3', '--max-train', '0'])
print(open(f'{W}/out/run/train_args.json').read())

In [ ]:
# 3. 評測：同一份權重，不掛 LoRA（原版）vs 掛 LoRA（微調版），和 Claude 寫的參考摘要比較
run([sys.executable, f'{W}/eval_sum.py', '--data', f'{W}/data', '--base', BASE, '--adapter', f'{W}/out/run/final',
     '--out', f'{W}/out/eval', '--batch-size', '4', '--load-4bit'])
rep = json.load(open(f'{W}/out/eval/sum_report.json', encoding='utf-8'))
names = {'json_ok': 'JSON 合法', 'quote_ok': '引用正確', 'count_diff': '項目數差距（越小越好）', 'topic_f1': '議題吻合',
         'action_f1': '待辦吻合', 'text_sim': '文字相似度', 'headline_sim': '一句話總結相似度'}
for grp in ['all', 'zh', 'en']:
    print(f'\n== {grp}（{rep["base"][grp]["n"]} 場）')
    for k, label in names.items():
        print(f'{label:<16} 原版 {rep["base"][grp][k]}   微調版 {rep["ft"][grp][k]}')

In [ ]:
# 4. 存檔：結果與 LoRA 盡量複製到雲端硬碟（雲端硬碟空間不夠也不影響上面已印出的結果）
try:
    from google.colab import drive
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    dst = '/content/drive/MyDrive/MeetSum_summary_ft'
    os.makedirs(dst, exist_ok=True)
    shutil.copytree(f'{W}/out/eval', f'{dst}/eval', dirs_exist_ok=True)
    shutil.copytree(f'{W}/out/run/final', f'{dst}/lora_final', dirs_exist_ok=True)
    for f in ['train_log.json', 'train_args.json']:
        shutil.copy(f'{W}/out/run/{f}', dst)
    drive.flush_and_unmount()
    print('✅ 已存到雲端硬碟 MeetSum_summary_ft/')
except Exception as e:
    print('⚠️ 存雲端硬碟失敗：', e)

In [ ]:
# 5. 關機（確認上面結果都看到之後才執行）
from google.colab import runtime
runtime.unassign()